# azroute: search + learning router for the M3D routing challenge

Runs the full training-and-solving loop on a Colab GPU runtime (built for **G4**, about 24 h):

* worker processes improve all 45 challenge cases (rip-up and reroute, guided by search over
  routing decisions) and record training data;
* the learner trains the network (multi-scale CNN + net interaction layers, ~10 M parameters) on the GPU and publishes it to the workers;
* everything is checkpointed to Google Drive; a disconnected session resumes where it left off;
* at the deadline it stops and exports checked submission files (`.sol.json` + `runtime.json` per tier).

**Use:** Runtime > Change runtime type > **G4** GPU. Fill in the settings, then run the cells top to bottom.
After a disconnect, run cells 1-5 again; the run resumes from Drive and keeps its original deadline.
To start over, use a new `DRIVE_DIR`.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/Dev077/azroute.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
CHALLENGE_URL = "https://github.com/partcleda/eda-3d-routing-challenge.git"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/azroute_run"  #@param {type:"string"}
HOURS = 23.5  #@param {type:"number"}
TIERS = "all"  #@param {type:"string"}
WORKERS = 44  #@param {type:"integer"}
#@markdown `WORKERS`: worker processes, `0` = auto (one per spare CPU, capped by RAM and GPU memory; ~3 GB RAM each).
#@markdown `TIERS`: `all` or a comma list of intro,hard,scale,stress,congested,designs.
#@markdown For a private repo, put a token in the URL: `https://<token>@github.com/you/azroute.git`.
WORK_DIR = "/content/work"
CODE_DIR = "/content/azroute"

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Get the code, build the router core, run the tests
import os, subprocess
os.chdir('/content')
def sh(cmd, cwd='/content'):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode: raise RuntimeError(f'failed: {cmd}')

if not os.path.isdir('/content/eda-3d-routing-challenge'):
    sh(f'git clone --depth 1 {CHALLENGE_URL} eda-3d-routing-challenge')
else:
    sh('git -C eda-3d-routing-challenge pull --ff-only')
if not os.path.isdir(CODE_DIR):
    sh(f'git clone -b {BRANCH} {REPO_URL} {CODE_DIR}')
else:
    sh(f'git -C {CODE_DIR} pull --ff-only')
os.environ['AZROUTE_CHALLENGE_ROOT'] = '/content/eda-3d-routing-challenge'
os.environ['PYTHONPATH'] = CODE_DIR
sh('pip -q install pybind11 psutil')
sh('python setup.py > build.log 2>&1 || (tail -40 build.log; exit 1)', cwd=CODE_DIR)
sh('python -m unittest tests.test_core', cwd=CODE_DIR)
print('ready:', CODE_DIR)

In [ ]:
#@title 4. Hardware
import torch, psutil
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('cpus:', os.cpu_count(), '| RAM GB:', round(psutil.virtual_memory().total / 2**30, 1))
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > G4'

In [ ]:
#@title 5. Start (or resume) the run in the background
import json, subprocess, os
# '[p]ipeline[.]run' matches the run's command line but not this check's own shell
running = subprocess.run("pgrep -f '[p]ipeline[.]run'", shell=True, capture_output=True, text=True).stdout.split()
if running:
    print('already running, pid', running)
else:
    os.makedirs(WORK_DIR, exist_ok=True)
    args = ['python', '-m', 'pipeline.run', '--work', WORK_DIR, '--drive', DRIVE_DIR, '--tiers', TIERS, '--workers', str(WORKERS)]
    prev = os.path.join(DRIVE_DIR, 'run_config.json')
    if not os.path.exists(prev):
        args += ['--hours', str(HOURS)]          # fresh run; on resume the stored deadline is kept
    logf = open(os.path.join(WORK_DIR, 'run.log'), 'a')
    p = subprocess.Popen(args, cwd=CODE_DIR, env=dict(os.environ), stdout=logf, stderr=subprocess.STDOUT,
                         start_new_session=True)
    print('started pid', p.pid, ' '.join(args))

In [ ]:
#@title 6. Monitor (re-run any time)
import json, math, os, collections
import matplotlib.pyplot as plt
print(''.join(open(os.path.join(WORK_DIR, 'run.log')).readlines()[-15:]))
st_path = os.path.join(WORK_DIR, 'case_stats.json')
if os.path.exists(st_path):
    stats = {k: v for k, v in json.load(open(st_path)).items() if not k.startswith('_')}
    by = collections.defaultdict(list)
    for k, s in stats.items():
        if s.get('best'):
            by[s['tier']].append(s)
    print(f"{'tier':10} {'cases':>5} {'vs baseline':>12} {'vs initial':>11} {'delay/LB':>9}")
    for t, rs in by.items():
        gm = math.exp(sum(math.log(s['baseline'] / s['best']) for s in rs) / len(rs))
        gi = math.exp(sum(math.log(s.get('initial_delay', s['best']) / s['best']) for s in rs) / len(rs))
        lb = sum(s['best'] for s in rs) / max(1, sum(s.get('lower_bound', 0) for s in rs))
        print(f"{t:10} {len(rs):5d} {gm:12.4f} {gi:11.4f} {lb:9.4f}")
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
mp = os.path.join(WORK_DIR, 'metrics.jsonl')
if os.path.exists(mp):
    m = [json.loads(l) for l in open(mp)]
    for key in ['policy_kl', 'value_mse', 'future_bce', 'blocked_bce']:
        ax[0].plot([r['step'] for r in m], [r[key] for r in m], marker='.', label=key)
    ax[0].set_title('training losses'); ax[0].set_xlabel('step'); ax[0].legend(); ax[0].set_yscale('log')
vp = os.path.join(WORK_DIR, 'visits.jsonl')
if os.path.exists(vp) and os.path.exists(st_path):
    v = sorted((json.loads(l) for l in open(vp)), key=lambda r: r['t'])
    t0 = v[0]['t']; best = {}; series = collections.defaultdict(list)
    for r in v:
        if 'delay' not in r: continue
        best[r['case']] = r['delay']
        tier = r['case'].split('/')[0]
        rs = [(stats[k]['baseline'], d) for k, d in best.items() if k.startswith(tier + '/')]
        series[tier].append(((r['t'] - t0) / 3600, math.exp(sum(math.log(b / d) for b, d in rs) / len(rs))))
    for tier, pts in series.items():
        ax[1].plot([p[0] for p in pts], [p[1] for p in pts], label=tier)
    ax[1].set_title('aggregate vs baseline (cases seen so far)'); ax[1].set_xlabel('hours'); ax[1].legend()
    # scored cases only, and only since the first network visit, so both bars cover the same hours
    net = [r for r in v if r.get('reroutes')]
    t_net = min((r['t'] for r in net if r.get('network')), default=None)
    for flag, lab in [(False, 'network-free (control)'), (True, 'network')]:
        rr = [r for r in net if r.get('network') == flag and t_net is not None and r['t'] >= t_net]
        if rr:
            ax[2].bar(lab, sum(r['better'] for r in rr) / max(1, sum(r['reroutes'] for r in rr)))
            ax[2].text(lab, 0, f" n={sum(r['reroutes'] for r in rr)}", rotation=90, va='bottom')
    ax[2].set_title('scored cases: share of reroutes that improve')
plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Stop early (optional): workers finish their current reroute, then it syncs and exports
open(os.path.join(WORK_DIR, 'STOP'), 'w').close()
print('stop requested; watch run.log in cell 6')
# To kill a run outright instead (no export), kill its whole process group - the workers have
# command lines without "pipeline.run" in them, so pkill on the name alone leaves them running:
#   !kill -TERM -- -$(pgrep -f '[p]ipeline[.]run' | head -1)


In [ ]:
#@title 8. Export checked submissions now (also happens automatically at the deadline)
out = os.path.join(DRIVE_DIR, 'export')
sh(f'python -m pipeline.export --work {WORK_DIR} --out {out}', cwd=CODE_DIR)
sh(f'cd {out} && zip -qr submissions.zip submissions summary.md && ls -la')
print('submission files in', os.path.join(out, 'submissions'))